# 새 킷 데이터에서 탐 규칙 검증 — GPU 서버 실행 (S15P31A307-164, 작업 19)

E-GMD test 어쿠스틱 26킷 1,170곡 · STAR Drums test 31곡 · ENST minus-one 64곡의 특징(채보 모델 프레임별 점수 + 드럼 킷 분리 트랙)을 `data/rule_features/`에 저장하고, 규칙 후보를 `rule-eval`로 비교한다. 특징 파일은 수십 GB라 서버에 두고 결과 폴더만 내려받는다. **위에서부터 차례로 실행한다.** 셀마다 명령만 부르고 로직은 없다.

**준비물.** 로컬 `AI/`에서 `uv run python scripts/pack_server_upload.py --code-only`로 만든 `data/upload/server/S15P31A307_code.zip`을 JupyterHub `ssaplash/upload/`에 올린다(파일 목록 화면 오른쪽 위 Upload). 이 노트북은 `ssaplash/notebooks/`에 올린다. 서버에는 지난 세션의 `.venv`·데이터 링크(`data/e-gmd`, `data/star_drums/full`)·`data/models`가 이미 있다.

**순서.** 1 설정 → 2 코드 덮어 풀기(1분) → 3 ENST 링크·데이터 점검(2~3분) → 4 특징 저장 시작(백그라운드, **한 번만**) → 5 진행 확인(여러 번 쳐도 됨, ENST·STAR 30분 안, E-GMD 3~4시간) → 6 규칙 비교(CPU, 10~20분) → 7 결과 묶기.

**끊기면.** 4번은 nohup이라 브라우저를 닫아도 끝까지 돈다. 커널이 다시 시작됐으면 1번 셀만 다시 실행하고 5번부터 이어 간다. 4번을 다시 치면 안 된다(이미 있는 곡은 건너뛰므로 망가지지는 않지만 두 개가 같이 돈다).

In [ ]:
# 1. 설정. 지정받은 GPU는 6번
import os
import sys

GPU_ID = "6"
os.environ["CUDA_VISIBLE_DEVICES"] = GPU_ID
os.environ["PYTHONUTF8"] = "1"
os.environ["MPLBACKEND"] = "Agg"
HOME = os.path.expanduser("~")
WORK = f"{HOME}/ssaplash"
AI = f"{WORK}/S15P31A307/AI"
PY = sys.executable
UPLOAD = f"{WORK}/upload"
RESULTS = f"{WORK}/results"
LOG = f"{AI}/rule_features_newkits.log"
os.makedirs(RESULTS, exist_ok=True)
!nvidia-smi -i $GPU_ID --query-gpu=index,name,memory.used,memory.total,utilization.gpu --format=csv
!df -h $HOME | tail -1
!ls -la $UPLOAD/S15P31A307_code.zip
# 확인: GPU 한 줄(H200), 디스크 여유 100GB 이상, 코드 zip이 오늘 날짜

In [ ]:
# 2. 코드 덮어 풀기. 기존 폴더를 지우지 않고 위에 덮어쓴다 (.venv·데이터 링크·특징 파일이 그대로 남음)
!$PY -m zipfile -e $UPLOAD/S15P31A307_code.zip $WORK
%cd $AI
!ls src/ssaplash_ai/bench/datasets/enst.py configs/bench/datasets/enst_minus_one.yaml docs/chart-rules/tom-newkits-2026-10-29/candidates.yaml
!grep -c stems-from src/ssaplash_ai/bench/cli.py
!.venv/bin/python -c "import torch, adtof_pytorch, msst; print('torch', torch.__version__, 'cuda', torch.cuda.is_available())"
# 확인: 파일 3개가 그대로 찍히고, grep 결과가 1 이상, 'cuda True'. 파일이 없다고 나오면 zip을 다시 만든 것인지 확인

In [ ]:
# 3. ENST 링크 + 데이터 점검 (여러 번 쳐도 됨)
!mkdir -p data/enst && ln -sfn $HOME/ssaplash/datasets/enst-drums/ENST-drums-public data/enst/ENST-drums-public
!ls data/enst/ENST-drums-public
!.venv/bin/ssaplash-bench check-data configs/bench/datasets/enst_minus_one.yaml configs/bench/datasets/star_drums_test.yaml configs/bench/datasets/egmd_test_acoustic.yaml
# 확인: ls에 drummer_1 drummer_2 drummer_3. check-data가 enst_minus_one 64곡·누락 0 (탐 3종 합 약 2,600),
#       star_drums_test 31곡·누락 0, egmd_test_acoustic 1170곡·누락 0. 누락이 있으면 그 줄을 적어 두고 멈춤

In [ ]:
# 4. 특징 저장 시작 (백그라운드, 한 번만). ENST → STAR → E-GMD 순서. 이미 있는 곡은 건너뜀
import subprocess

assert not os.path.exists(LOG), "이미 시작했음. 5번 셀로 진행을 확인한다"
cmd = [
    ".venv/bin/ssaplash-bench", "rule-features",
    "--dataset",
    "configs/bench/datasets/enst_minus_one.yaml",
    "configs/bench/datasets/star_drums_test.yaml",
    "configs/bench/datasets/egmd_test_acoustic.yaml",
]
proc = subprocess.Popen(
    ["nohup", *cmd], cwd=AI, stdout=open(LOG, "w"), stderr=subprocess.STDOUT,
    env=dict(os.environ), start_new_session=True,
)
print("시작. pid", proc.pid, "로그", LOG)
# 확인: '시작. pid 숫자'. 바로 5번으로

In [ ]:
# 5. 진행 확인 (여러 번 쳐도 됨). 곡 수가 64 / 31 / 1170이 되면 끝
!for d in enst_minus_one star_drums_test egmd_test_acoustic; do echo "$d: $(ls data/rule_features/$d 2>/dev/null | grep -c npz)개"; done
!tail -3 $LOG
!grep -c '실패 [1-9]' $LOG || true
!pgrep -f "[s]saplash-bench rule-features" > /dev/null && echo "아직 도는 중" || echo "끝났거나 안 돌고 있음"
!du -sh data/rule_features
# 확인: 세 줄의 개수가 늘어나고 마지막에 '[rule-features] 곡이름: 탐 N·심벌 M (초)'. '실패' 수가 0이어야 함.
#       '끝났거나 안 돌고 있음'인데 개수가 64/31/1170에 못 미치면 로그 끝의 오류를 적어 두고 4번을 한 번 더 (있는 곡은 건너뜀)

In [ ]:
# 6. 규칙 비교 (CPU, 특징 저장이 다 끝난 뒤). 후보 5개 + 탐 기준값 40개. 탐 둘인 곡의 위쪽 탐은 미드로 채점
DS = "configs/bench/datasets/enst_minus_one.yaml configs/bench/datasets/star_drums_test.yaml configs/bench/datasets/egmd_test_acoustic.yaml"
!.venv/bin/ssaplash-bench rule-eval --dataset $DS --candidates docs/chart-rules/tom-newkits-2026-10-29/candidates.yaml --out data/results/rule-eval-newkits --two-tom-upper MID_TOM --mlflow-uri none
!.venv/bin/ssaplash-bench rule-eval --dataset $DS --candidates docs/chart-rules/tom-newkits-2026-10-29/candidates_threshold.yaml --out data/results/rule-eval-newkits-threshold --two-tom-upper MID_TOM --mlflow-uri none
!head -20 data/results/rule-eval-newkits/summary.md
# 확인: '[rule-eval] 후보 · 데이터셋: f1Tom=...' 줄이 후보마다 세 줄씩, 끝에 '후보 5개 × 묶음 3개 → ...summary.md'. E-GMD는 1,170곡이라 후보 하나에 몇 분

In [ ]:
# 7. 결과 묶기. results 폴더의 rule_eval_newkits.zip 을 내려받아 로컬 AI/data/results/ 에 둔다 (푸는 것은 세션에서. zip 안 경로가 AI/ 기준)
!rm -f $RESULTS/rule_eval_newkits.zip
!$PY -m zipfile -c $RESULTS/rule_eval_newkits.zip data/results/rule-eval-newkits data/results/rule-eval-newkits-threshold $LOG
!ls -la $RESULTS/rule_eval_newkits.zip
# 확인: 몇 MB짜리 zip 하나